In [ ]:
import os
import json
import random
import inspect
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)


In [ ]:
MODEL_NAME = "roberta-base"
MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8

NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
DROPOUT_RATE = 0.3
WARMUP_RATIO = 0.1
SEED = 42

TRAIN_URL = "https://raw.githubusercontent.com/UKPLab/emnlp2024-llm-classifier/main/data/Re3-Sci/tasks/edit_intent_classification/train.csv"
VAL_URL = "https://raw.githubusercontent.com/UKPLab/emnlp2024-llm-classifier/main/data/Re3-Sci/tasks/edit_intent_classification/val.csv"
TEST_URL = "https://raw.githubusercontent.com/UKPLab/emnlp2024-llm-classifier/main/data/Re3-Sci/tasks/edit_intent_classification/test.csv"

OUTPUT_DIR = "snet_strong_outputs"
snet_final_model = "best_snet_strong_model"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(snet_final_model, exist_ok=True)


In [ ]:
# Check for GPU
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Using device:", device)


In [ ]:
# Load Data and Preprocessing
train_df = pd.read_csv(TRAIN_URL)
val_df = pd.read_csv(VAL_URL)
test_df = pd.read_csv(TEST_URL)

def safe_text(value):
    if pd.isna(value): 
        return ""
    return str(value).strip()

for df in [train_df, val_df, test_df]:
    df["src_text"] = df["text_src"].apply(safe_text)
    df["tgt_text"] = df["text_tgt"].apply(safe_text)

labels = sorted(train_df["label"].unique().tolist())
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

for df in [train_df, val_df, test_df]:
    df["label_id"] = df["label"].map(label2id)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("Labels:", label2id)

In [ ]:
# Weights

train_label_ids = train_df["label_id"].values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(sorted(np.unique(train_label_ids))),
    y=train_label_ids
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights)

In [ ]:
# Datasets

train_ds = Dataset.from_pandas(
    train_df[["src_text", "tgt_text", "label_id"]],
    preserve_index=False
)
val_ds = Dataset.from_pandas(
    val_df[["src_text", "tgt_text", "label_id"]],
    preserve_index=False
)
test_ds = Dataset.from_pandas(
    test_df[["src_text", "tgt_text", "label_id"]],
    preserve_index=False
)

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# Tokenization

def tokenize_snet(batch):
    src_encoded = tokenizer(
        batch["src_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    tgt_encoded = tokenizer(
        batch["tgt_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    return {
        "src_input_ids": src_encoded["input_ids"],
        "src_attention_mask": src_encoded["attention_mask"],
        "tgt_input_ids": tgt_encoded["input_ids"],
        "tgt_attention_mask": tgt_encoded["attention_mask"],
        "labels": batch["label_id"]
    }

train_tokenized = train_ds.map(tokenize_snet, batched=True, batch_size=1000)
val_tokenized = val_ds.map(tokenize_snet, batched=True, batch_size=1000)
test_tokenized = test_ds.map(tokenize_snet, batched=True, batch_size=1000)

columns_to_keep = [
    "src_input_ids",
    "src_attention_mask",
    "tgt_input_ids",
    "tgt_attention_mask",
    "labels"
]

train_tokenized.set_format(type="torch", columns=columns_to_keep)
val_tokenized.set_format(type="torch", columns=columns_to_keep)
test_tokenized.set_format(type="torch", columns=columns_to_keep)

In [ ]:
# Snet Model

class Snet_Model_(nn.Module):
    def __init__(self, model_name, num_labels, dropout_rate=0.3, class_weights=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.encoder.gradient_checkpointing_enable()
        hidden_size = self.encoder.config.hidden_size

        # [target, |target-source|, source]
        feature_size = hidden_size * 3
        self.dropout1 = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(feature_size, hidden_size)
        self.norm1 = nn.LayerNorm(hidden_size)
        self.act1 = nn.GELU()
        self.dropout2 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.norm2 = nn.LayerNorm(hidden_size // 2)
        self.act2 = nn.GELU()
        self.dropout3 = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(hidden_size // 2, num_labels)
        self.class_weights = class_weights

    def encode_text(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return cls_embedding

    def forward(
        self,
        src_input_ids=None,
        src_attention_mask=None,
        tgt_input_ids=None,
        tgt_attention_mask=None,
        labels=None
    ):
        source_embedding = self.encode_text(src_input_ids, src_attention_mask)
        target_embedding = self.encode_text(tgt_input_ids, tgt_attention_mask)
        diff_abs = torch.abs(target_embedding - source_embedding)

        # n-diffABS-o
        features = torch.cat([target_embedding, diff_abs, source_embedding],   dim=1  )

        x = self.dropout1(features)
        x = self.fc1(x)
        x = self.norm1(x)
        x = self.act1(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        x = self.norm2(x)
        x = self.act2(x)
        x = self.dropout3(x)
        logits = self.classifier(x)

        loss = None
        if labels is not None:
            if self.class_weights is not None:
                loss_fn = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            else:
                loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits
        }

model = Snet_Model_(
    model_name=MODEL_NAME,
    num_labels=len(labels),
    dropout_rate=DROPOUT_RATE,
    class_weights=class_weights
)
model.to(device)

In [ ]:
# Metrics

def compute_metrics(eval_pred):
    logits, true_labels = eval_pred
    pred_labels = np.argmax(logits, axis=1)
    accuracy = accuracy_score(true_labels, pred_labels)
    precision, recall, f1, _ = precision_recall_fscore_support( true_labels, pred_labels, average="weighted", zero_division=0 )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
# Training Arguments

def build_training_args():
    signature = inspect.signature(TrainingArguments.__init__)
    valid_params = set(signature.parameters.keys())

    args_dict = {
        "output_dir": OUTPUT_DIR,
        "num_train_epochs": NUM_EPOCHS,
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "greater_is_better": True,
        "save_total_limit": 2,
        "fp16": torch.cuda.is_available(),
        "seed": SEED,
        "max_grad_norm": 1.0
    }

    if "evaluation_strategy" in valid_params:
        args_dict["evaluation_strategy"] = "epoch"
    elif "eval_strategy" in valid_params:
        args_dict["eval_strategy"] = "epoch"

    if "save_strategy" in valid_params:
        args_dict["save_strategy"] = "epoch"

    if "logging_strategy" in valid_params:
        args_dict["logging_strategy"] = "epoch"

    if "report_to" in valid_params:
        args_dict["report_to"] = "none"

    if "dataloader_num_workers" in valid_params:
        args_dict["dataloader_num_workers"] = 0

    if "dataloader_pin_memory" in valid_params:
        args_dict["dataloader_pin_memory"] = torch.cuda.is_available()

    if "overwrite_output_dir" in valid_params:
        args_dict["overwrite_output_dir"] = True

    return TrainingArguments(**args_dict)

training_args = build_training_args()

In [ ]:
# Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

train_result = trainer.train()
print(train_result)

In [ ]:
# Evaluation

test_metrics = trainer.evaluate(test_tokenized)
print("\nTest metrics:")
print(test_metrics)

prediction_output = trainer.predict(test_tokenized)
pred_ids = np.argmax(prediction_output.predictions, axis=1)
true_ids = prediction_output.label_ids

pred_names = [id2label[i] for i in pred_ids]
true_names = [id2label[i] for i in true_ids]

print("\nClassification Report:")
print(classification_report(true_names, pred_names, labels=labels, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(true_names, pred_names, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)


In [ ]:
# Model Save

trainer.save_model(snet_final_model)
tokenizer.save_pretrained(snet_final_model)

with open(os.path.join(snet_final_model, "label_mapping.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "label2id": label2id,
            "id2label": {str(k): v for k, v in id2label.items()}
        },
        f,
        indent=4
    )

with open(os.path.join(snet_final_model, "test_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=4)

cm_df.to_csv(os.path.join(snet_final_model, "confusion_matrix.csv"))

print("\nModel Saved:", snet_final_model)